# Agent | Tools

**1. 도구 (Tool)**

https://docs.langchain.com/oss/python/integrations/tools/index#tools-and-toolkits

> "LLM이 사용할 수 있는 구체적인 기술이나 장비"

LLM은 기본적으로 학습된 데이터 내에서만 답변할 수 있으며, 실시간 정보나 정확한 수학 계산에는 취약하다. **Tool**은 이러한 한계를 보완하기 위해 LLM에게 쥐여주는 외부 기능이다.

* **역활:** 외부 API 호출, 웹 검색, 코드 실행, 파일 시스템 접근 등 LLM이 직접 할 수 없는 작업을 대신 수행한다.
* **예시:**
* `Google Search`: 최신 정보를 검색한다.
* `Calculator`: 정확한 수학 계산을 수행한다.
* `Python REPL`: 파이썬 코드를 작성하고 실행한다.



**2. 에이전트 (Agent)**

> "도구를 언제, 어떻게 사용할지 결정하는 두뇌"

**Agent**는 LLM을 추론 엔진(Reasoning Engine)으로 사용하여 사용자의 요청을 해결하기 위한 계획을 세우고 실행하는 주체이다. 단순히 정해진 코드를 순서대로 실행하는 것이 아니라, 상황에 따라 유연하게 행동을 결정한다.

* **역활:** 사용자의 질문을 분석하고, 어떤 **Tool**이 필요한지 판단(Thought)하고, 해당 도구를 실행(Action)한 뒤, 그 결과(Observation)를 보고 다음 행동을 결정하거나 최종 답변을 내놓는다.
* **작동 방식 (ReAct 패턴 예시):**
1. **질문:** "현재 서울 날씨에 맞는 옷차림 추천해줘."
2. **생각(Thought):** "서울의 현재 날씨를 먼저 알아야 한다." -> `Search` 도구 선택
3. **행동(Action):** `Search("서울 현재 날씨")` 실행
4. **관찰(Observation):** "서울 기온 5도, 맑음"이라는 결과 획득
5. **생각(Thought):** "5도면 코트나 패딩이 필요하다."
6. **최종 답변:** "현재 서울은 5도이므로 코트나 가벼운 패딩을 추천합니다."

**Agent와 Tools의 상호작용**

1. **Agent가 입력을 받음**: 사용자의 요청을 LLM으로 분석.
2. **적합한 Tool 선택**: 요청을 처리하는 데 가장 적합한 Tool을 선택.
3. **Tool 실행 및 결과 반환**: Tool을 실행하고 결과를 받아 사용자에게 응답.

In [1]:
%pip install -Uqqq langchain_openai langchain_community langchain_tavily langgraph wikipedia numexpr arxiv ddgs

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://apac.api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

## Tools

In [3]:
import importlib, pkgutil  # 모듈 동적 로드 / 패키지 탐색 유틸

# langchain_community.tools 패키지 로드
package = importlib.import_module('langchain_community.tools')

# 해당 패키지 경로 아래의 하위 모듈들을 하나씩 순회
for module in pkgutil.iter_modules(package.__path__):
    print(module.name)  # 각 모듈(도구) 이름 출력

ainetwork
amadeus
arxiv
asknews
audio
azure_ai_services
azure_cognitive_services
bearly
bing_search
brave_search
cassandra_database
clickup
cogniswitch
connery
convert_to_openai
databricks
dataforseo_api_search
dataherald
ddg_search
e2b_data_analysis
edenai
eleven_labs
few_shot
file_management
financial_datasets
github
gitlab
gmail
golden_query
google_books
google_cloud
google_finance
google_jobs
google_lens
google_scholar
google_serper
google_trends
graphql
human
ifttt
interaction
jina_search
jira
json
memorize
merriam_webster
metaphor_search
mojeek_search
multion
nasa
nuclia
office365
openai_dalle_image_generation
openapi
openweathermap
passio_nutrition_ai
playwright
plugin
polygon
powerbi
pubmed
render
requests
riza
scenexplain
searchapi
searx_search
semanticscholar
shell
slack
sleep
spark_sql
sql_database
stackexchange
steam
steamship_image_generation
tavily_search
vectorstore
wikidata
wikipedia
wolfram_alpha
yahoo_finance_news
you
youtube
zapier
zenguard


### Wikipedia Tool

In [4]:
from langchain_community.tools import WikipediaQueryRun        # 위키피디아 질문 실행 Tool
from langchain_community.utilities import WikipediaAPIWrapper  # 위키피디아 검색/요약 API 요청 래퍼 클래스

# 위키피디아 API 래퍼를 Tool에 연결
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
# print(wiki_tool.run('Langchain Tools'))  # 위키피디아 검색/요약 결과 출력

In [5]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pprint import pprint

messages = [('human', '한국 신생그룹 롱샷의 멤버 알려줘')]

llm = init_chat_model('gpt-5.4-mini')
# print(llm.invoke('걸그룹 튜이드 멤버 알려줘'))  # 최신정보 알지 못함.

agent = create_agent(
    model = llm,
    tools = [wiki_tool]
)

response = agent.invoke({'messages': messages})

pprint(response)

{'messages': [HumanMessage(content='한국 신생그룹 롱샷의 멤버 알려줘', additional_kwargs={}, response_metadata={}, id='0c3da638-b8ea-404a-9e59-90b375cc7ec6'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 177, 'total_tokens': 204, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeTwARNwdUlwGNCX6yiFPxRsqJi3', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b5-2475-75e1-b8ed-9afc03a2731d-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': '한국 신생그룹 롱샷 멤버'}, 'id': 'call_pNd5oyyHNkpg1KeZA74Tb76s

In [6]:
print(response['messages'][-1].content)

죄송하지만 제가 가진 정보로는 **한국 신생그룹 “롱샷”의 멤버를 확인할 수 없습니다.**

원하시면 제가 대신:
- **최근 기사/공식 SNS 기준으로 확인하는 방법**을 알려드리거나
- 그룹 이름의 **영문 표기**가 있다면 다시 찾아볼 수 있어요.

혹시 **“롱샷”의 정확한 영문명**이나 소속사 정보를 알면 더 정확하게 도와드릴게요.


### load_tools

**load_tools 사용가능 목록**

라이브러리를 통해 제공되는 외부 tool들을 langchain-community에서 통합하여 사용할 수 있다.
웬만한 기능들은 langchain 생태계 내에서 쉽게 쓸 수 있다.

https://docs.langchain.com/oss/python/integrations/providers/overview

https://docs.langchain.com/oss/python/integrations/tools

| 도구 이름        | 기능 예시               |
|------------------|------------------------|
| llm-math         | LLM 기반 수학 계산     |
| wikipedia        | 위키백과 검색          |
| serpapi          | 구글 검색 API          |
| requests_get     | HTTP GET 요청          |
| requests_post    | HTTP POST 요청         |
| arxiv            | arXiv 논문 검색        |
| pubmed           | PubMed 논문 검색       |
| dalle            | DALL-E 이미지 생성     |
| bing_search      | Bing 검색              |
| duckduckgo_search| DuckDuckGo 검색        |

### arxiv

In [7]:
import requests  # HTTP 요청 보내는 라이브러리
import xml.etree.ElementTree as ET  # XML 응답 파싱
from langchain_core.tools import tool  # Langchain Tool 생성 데코레이터

@tool
def search_arxiv(arxiv_id: str) -> str:
    """ arXiv 논문 ID로 제목, 저자, 초록을 조회합니다. """

    url = "https://export.arxiv.org/api/query"
    response = requests.get(
        url,
        params = {
            "id_list": arxiv_id,  # 논문 ID
            "max_results": 1  # 결과 1개
        },
        timeout = 10  # 응답 대기시간
    )

    response.raise_for_status()  # 요청 실패시 예외 발생

    root = ET.fromstring(response.text)  # XML 문자열을 받아 Element 객체로 변환

    ns = {"atom": "http://www.w3.org/2005/Atom"}  # arXiv 응답의 XML 네임스페이스
    entry = root.find("atom:entry", ns)  # 논문 정보가 담긴 entry 태그

    if entry is None:
        return "논문 정보를 찾을 수 없습니다."

    title = entry.findtext("atom:title", namespaces=ns).strip()  # 논문 제목 추출
    summary = entry.findtext("atom:summary", namespaces=ns).strip()  # 요약 정보 추출
    authors = [  # 저자들 추출
        author.findtext("atom:name", namespaces=ns)
        for author in entry.findall("atom:author", ns)
    ]

    return f"""
제목: {title}
저자: {', '.join(authors)}
초록: {summary}
"""

In [8]:
tools = [search_arxiv, wiki_tool]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = "당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요."
)

messages = [('human', '1706.03762 이 논문의 내용을 간단하게 설명해줄래? (한글답변)')]
response = agent.invoke({'messages': messages})
pprint(response['messages'][-1].content)

('물론입니다. **1706.03762는 유명한 논문 _“Attention Is All You Need”_**로, '
 '**Transformer**라는 새로운 딥러닝 모델을 처음 제안한 논문입니다.\n'
 '\n'
 '### 핵심 내용\n'
 '- 기존의 번역 모델은 주로 **RNN/LSTM**이나 **CNN**을 사용했는데, 이 논문은 이를 버리고 **Attention '
 '메커니즘만으로** 모델을 구성했습니다.\n'
 '- 이 구조를 통해 문장 전체의 단어 관계를 **한 번에 병렬적으로 처리**할 수 있어서,\n'
 '  - **학습이 더 빠르고**\n'
 '  - **성능도 더 좋다**는 것을 보였습니다.\n'
 '\n'
 '### 왜 중요한가?\n'
 '- 이전 모델은 문장을 순서대로 처리해야 해서 느렸지만,\n'
 '- Transformer는 **Self-Attention**으로 각 단어가 문장 내 다른 단어들과 어떻게 연결되는지 직접 살펴봅니다.\n'
 '- 이 방식이 이후 **BERT, GPT, T5** 같은 많은 최신 언어 모델의 기반이 되었습니다.\n'
 '\n'
 '### 논문 결과\n'
 '- 영어-독일어 번역, 영어-프랑스어 번역 등에서 기존 최고 성능을 뛰어넘었습니다.\n'
 '- 특히 **적은 학습 비용으로도 높은 번역 성능**을 보였습니다.\n'
 '\n'
 '### 한 줄 요약\n'
 '**“문장을 순차적으로 처리하는 RNN 대신, Attention만으로도 더 빠르고 효과적으로 번역할 수 있다”**는 것을 보여준 매우 '
 '중요한 논문입니다.\n'
 '\n'
 '원하시면 제가 이 논문의 **Transformer 구조(Encoder/Decoder, Self-Attention)**도 아주 쉽게 '
 '그림처럼 설명해드릴게요.')


### llm-math

In [9]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-4.1-mini')

# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm 필요)
tools = load_tools(['wikipedia', 'llm-math'], llm=llm)

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = """
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해 주세요.
단, 숫자계산은 반드시 llm-math 도구를 사용해서 답변에 활용해야 합니다.
"""
)

response = agent.invoke({'messages': '3.5의 3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘.'})
pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='3.5의 3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘.', additional_kwargs={}, response_metadata={}, id='cfe67a2a-d3b1-4c15-be6e-b9f85bb358d5'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 190, 'total_tokens': 210, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c721606bd7', 'id': 'chatcmpl-EHeU4QdYTuKkNXRvHyIdhr9fqbV9P', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b5-4b45-7c91-a37d-050ee695a434-0', tool_calls=[{'name': 'Calculator', 'args': {'__arg1': '3.5^3'}, 'id': 'call_UrWJE

### duckduckgo
https://reference.langchain.com/python/langchain-community/tools/ddg_search/tool/DuckDuckGoSearchRun

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

In [10]:
# 덕덕고 검색 Tool 2종류
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

ddgs = DuckDuckGoSearchRun()  # 검색 결과를 텍스트 요약 형태로 반환
print(ddgs.invoke("Trump's first name?"))  # 문자열 출력

ddgs2 = DuckDuckGoSearchResults()  # 검색 결과를 제목/링크/스니펫 형태로 반환
print(ddgs2.invoke("Trump's first name?"))  #  결과 출력

Donald Trump - Wikipedia First presidency of Donald Trump - Wikipedia Trump was sworn in as president on January 20, 2017. During his first term, his administration focused on immigration, trade, tax cuts, and reducing government regulations. Trump withdrew the United States from the Trans-Pacific Partnership and announced that the country would leave the Paris Agreement on climate change. [13][14] He supported building a wall along the U.S.-Mexico border and ... If you feel that a man's real last name is whatever last name his ancestors used first, then Trump's real last name might be Drumpf. Former President Trump was born Donald John Trump to parents Fred and Mary Trump on June 14, 1946. The outspoken politician rarely acknowledges the biblical origins of his middle name, which is ironic considering that he now sells customized "God Bless the USA" bibles. Most of the time, when Trump incorporates his middle name, he does so by only using the first initial. However, in 2020, he ...
s

In [11]:
llm = init_chat_model('gpt-5.4-mini')

tools = [ddgs2]

agent = create_agent(llm, tools, system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.')

response = agent.invoke({'messages': 'gs25 민음사 빵'})

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='gs25 민음사 빵', additional_kwargs={}, response_metadata={}, id='0379825d-b703-4c92-bdff-9373850df573'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 202, 'total_tokens': 229, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeUGMyeGYlVsTMU2eNqWcP4tmAjC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b5-762b-7fe1-ba0b-baab7831b380-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'GS25 민음사 빵'}, 'id': 'call_ulaMEzxwvGgBL9aUlfJnC

### tavily-search

https://docs.langchain.com/oss/python/integrations/tools/tavily_search

In [12]:
from langchain_tavily import TavilySearch  # Tavily 검색 Tool

tavily_tool = TavilySearch(
    max_results = 3,
    topic = 'general',  # general/news/finance 등 선택
    include_images = True,  # 이미지 URL 함께 반환
    search_depth = 'advanced'  # basic/advanced (advanced는 더 깊게 찾음)
)

tavily_tool.invoke('2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?')

{'query': '2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?',
 'follow_up_questions': None,
 'answer': None,
 'images': ['https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426f67d1b5fa839e454dfe_79_thumbnail2.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426e7fc594ca778cc36ab8_79_2-1.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426ed2d1b5fa839e44e068_79_2-2.png',
  'https://lookaside.fbsbx.com/lookaside/crawler/threads/DcNY3WAE9AY/0/image.jpg',
  'https://cdn.wakeupnews.co.kr/news/photo/202601/973_1856_5154.png'],
 'results': [{'url': 'https://www.newsshin.co.kr/news/articleView.html?idxno=',
   'title': "【뉴스신ㅣ2026년 8월 22일(토) ㅣ대한민국 '핫' 이슈】",
   'content': ": 【뉴스신】2026년 8월 22일 '핫' 이슈】 주택 문제와 공급 확대, 청년에게는 진입 장벽이고, 기성세대에게는 자산 문제다.",
   'score': 0.902481,
   'raw_content': None,
   'images': [],
   'id': 'eb0227-00'},
  {'url': 'https://highyon.tistory.com/entry/%EB%8F%88-%EB%B2%84%EB%8A%94-%ED%95%AB%EC%9D%B4%EC%8A%88-2026%EB%85%84-8%EC%9B%94-

In [13]:
llm = init_chat_model('gpt-5.4-mini')

tools = [tavily_tool]

agent = create_agent(llm, tools, system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.')

response = agent.invoke({'messages': '현재 AI업계에서 가장 핫한 주제가 뭐야?'})

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='현재 AI업계에서 가장 핫한 주제가 뭐야?', additional_kwargs={}, response_metadata={}, id='cd8ab139-2049-4a04-bc8f-df44c85a53a6'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 1322, 'total_tokens': 1380, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeUNIIW3K2QIGwdnzde1g70CF6gq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b5-96b3-76c1-8ccd-b1373cfc58ad-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '현재 AI 업계 가장 핫한 주제 2026 latest trends AI in

In [14]:
llm = init_chat_model('gpt-5.4-mini')

tools = [tavily_tool]

agent = create_agent(llm, tools, system_prompt='''
당신은 미국주식시장 분석봇입니다.
사용자가 요청한 기업에 대한 2026년 보고서를 직관적으로 분석해주세요.

# 출력형식
다음 내용을 포함해 표형식 출력 (분석기관별 레코드로 작성)

1. 분석기관명
2. 목표주가범위 (최저 ~ 최대)
3. 전망근거 키워드
4. 신뢰도 지수(1 ~ 10)
''')

response = agent.invoke({'messages': '2026년 애플 주가 전망 분석해 줘'})

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='2026년 애플 주가 전망 분석해 줘', additional_kwargs={}, response_metadata={}, id='47e00043-8059-43dd-9d9e-473ee6a95bd9'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 1380, 'total_tokens': 1434, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeUWKfNav9kqq2sJ9HqfmZtivVqx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b5-b94d-7661-b7c0-f6be647c70b8-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Apple 2026 price target forecast analyst repo

In [15]:
from IPython.display import display, Markdown

# 마지막 메시지 내용을 Markdown 변환 후 보기 좋게 출력
display(Markdown(response['messages'][-1].content))

| 분석기관명 | 목표주가범위 (최저 ~ 최대) | 전망근거 키워드 | 신뢰도 지수(1~10) |
|---|---:|---|---:|
| Citi | $365 ~ $365 | 강한 iPhone 수요, 스마트폰 점유율 확대, 서비스 매출 성장, AI 기대감 | 8 |
| Morgan Stanley | $315 ~ $315 | 2027년 EPS 기대치 상향, 아이폰/생태계 방어력, 메모리 비용 부담 존재 | 8 |
| Goldman Sachs | $330 ~ $340 | 견조한 제품 수요, 비용 통제, 자사주 매입 확대, 마진 안정성 | 7 |
| Wedbush | $350 ~ $400 | Apple Intelligence, Siri 개선, AI 수익화, WWDC/신제품 모멘텀 | 7 |
| Jefferies | $283 ~ $283 | Q1 2026 아이폰 판매 호조, 평균판매가격(ASP) 방어, 메모리 원가 압박 완화 | 6 |
| Rosenblatt | $268 ~ $268 | 보수적 시각, Siri AI 지연, 중국 경쟁, 단기 밸류에이션 부담 | 6 |

**한줄 해석:**  
2026년 애플은 대체로 **상승 여력은 있으나, AI 수익화 속도와 중국 수요, 메모리 비용이 주가 상단을 결정**할 가능성이 큽니다. 시장 컨센서스는 대체로 **$300대 중후반까지의 우상향**에 무게를 두고 있습니다.

### @tool

In [16]:
# eval / exec로 문자열 코드 실행
a = 10
print(eval("5 + 3 + a"))  # 문자열을 평가해서 결과를 반환
exec("b = 10")            # 문자열을 실행 (할당 가능)
print(b)

18
10


In [17]:
from langchain_core.tools import tool

@tool
def simple_calculator(query: str) -> str:
    """
    산술연산을 위한 간단한 계산기 Tool
    Args:
        query: 계산식
    Return:
        계산식 결과값

    Examples:
    - simple_calculator("5 + 3 - 2") -> "계산 결과: 6"
    - simple_calculator("4 ** 2 / 8") -> "계산 결과: 2"
    """
    try:
        result = eval(query)           # 문자열을 eval로 평가(결과 반환)
        return f"계산 결과 : {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"

simple_calculator

StructuredTool(name='simple_calculator', description='산술연산을 위한 간단한 계산기 Tool\nArgs:\n    query: 계산식\nReturn:\n    계산식 결과값\n\nExamples:\n- simple_calculator("5 + 3 - 2") -> "계산 결과: 6"\n- simple_calculator("4 ** 2 / 8") -> "계산 결과: 2"', args_schema=<class 'langchain_core.utils.pydantic.simple_calculator'>, func=<function simple_calculator at 0x00000280594A1BC0>)

In [18]:
llm = init_chat_model('gpt-5.4-mini')

tools = [simple_calculator]

agent = create_agent(llm, tools, system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.')

response = agent.invoke({'messages': '7 + 3 * 8 이거를 계산해 줘.'})

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='7 + 3 * 8 이거를 계산해 줘.', additional_kwargs={}, response_metadata={}, id='0b038522-e91f-4be9-93e7-b9e8693cf2ed'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 250, 'total_tokens': 274, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeUoxrvVRFvxSe97qtQsUYqtXC88', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b5-fcaf-7843-bace-e27ed95e86eb-0', tool_calls=[{'name': 'simple_calculator', 'args': {'query': '7 + 3 * 8'}, 'id': 'call_kkE5Arl6b530kag8v6

In [19]:
response = agent.invoke({'messages': '김치볶음밥 레시피?'})

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='김치볶음밥 레시피?', additional_kwargs={}, response_metadata={}, id='cd3457a7-3b56-46fb-92e8-10631c02fe24'),
              AIMessage(content='물론이죠. 아주 기본적이고 맛있는 **김치볶음밥 레시피** 알려드릴게요.\n\n### 재료 (1~2인분)\n- 밥 1~2공기\n- 김치 1컵 정도\n- 돼지고기/스팸/참치 중 하나(선택)\n- 양파 1/4개\n- 대파 조금\n- 식용유 1~2큰술\n- 고춧가루 1/2큰술(선택)\n- 간장 1작은술\n- 설탕 1/2작은술\n- 참기름 1작은술\n- 김치국물 1~2큰술\n- 계란 1개\n- 깨소금 약간\n\n### 만드는 법\n1. **재료 손질**\n   - 김치는 잘게 썰고, 양파와 대파도 썰어주세요.\n   - 고기나 스팸을 넣는다면 먹기 좋게 잘라주세요.\n\n2. **볶기**\n   - 팬에 식용유를 두르고 대파를 먼저 볶아 향을 냅니다.\n   - 양파, 고기/스팸을 넣고 함께 볶아주세요.\n   - 김치를 넣고 2~3분 정도 충분히 볶습니다.\n\n3. **양념 넣기**\n   - 김치국물, 간장, 설탕, 고춧가루를 넣고 섞어줍니다.\n\n4. **밥 넣기**\n   - 밥을 넣고 잘 풀어가며 볶습니다.\n   - 밥이 고르게 섞이면 마지막에 참기름을 넣고 한 번 더 볶아주세요.\n\n5. **마무리**\n   - 접시에 담고 계란후라이를 올리면 완성!\n   - 깨소금도 뿌리면 더 맛있어요.\n\n### 팁\n- 김치가 너무 신맛이 강하면 설탕을 조금 더 넣으면 좋아요.\n- 밥은 **찬밥**이 더 볶기 좋습니다.\n- 더 고소하게 먹고 싶으면 치즈를 올려도 맛있어요.\n\n원하시면 제가 **스팸 김치볶음밥 버전**이나 **참치 김치볶음밥 버전**으로도 바로 적어드릴게요.', additional_kwargs={'refusal': None}, respo

In [20]:
import json

OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

@tool
def get_current_weather(city="Seoul", units="metric"):
    """
    OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**
            - 변환예시:
                - 서울 -> Seoul
                - 충남, 충청남도 -> Chungcheongnam-do
                - 부산 -> Busan
        - units: str 온도단위를 설정하는 문자열
          - metric(기본값: 섭씨, 미터)
          - imperial(화씨, 야드)
    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    """

    url = f'https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json()  # json -> dict

    weather_info = {}

    if response.status_code == 200:  # 정상 응답 받은 경우
        weather_description = data['weather'][0]['description']  # 날씨 설명
        temp = data['main']['temp']  # 현재 기온
        temp_feels_like = data['main']['feels_like']  # 체감 온도
        humidity = data['main']['humidity']  # 습도

        weather_info = {
            'city': city,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }

    else:  # 응답 불량
        weather_info = {
            'city': city,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like': 'Not Found',
            'humidity': 'Not Found'
        }

    return json.dumps(weather_info)  # dict -> json

get_current_weather

StructuredTool(name='get_current_weather', description='OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수\n\nArgs:\n    - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**\n        - 변환예시:\n            - 서울 -> Seoul\n            - 충남, 충청남도 -> Chungcheongnam-do\n            - 부산 -> Busan\n    - units: str 온도단위를 설정하는 문자열\n      - metric(기본값: 섭씨, 미터)\n      - imperial(화씨, 야드)\nReturn:\n    - str: json 형식으로 변환된 현재 날씨 정보', args_schema=<class 'langchain_core.utils.pydantic.get_current_weather'>, func=<function get_current_weather at 0x00000280594A3560>)

In [21]:
llm = init_chat_model('gpt-5.4-mini')

tools = [simple_calculator, get_current_weather]

agent = create_agent(llm, tools, system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.')

response = agent.invoke({'messages': '오늘 뭐 입어야 돼? 나 서울에서 일해.'})

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='오늘 뭐 입어야 돼? 나 서울에서 일해.', additional_kwargs={}, response_metadata={}, id='d7672262-00da-44c4-b1a1-56447a7974d2'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 417, 'total_tokens': 440, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeUtdvxVZyZVOoxzG2bfVYsXAYWY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b6-1086-7e81-9467-7dbb401f3bda-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Seoul', 'units': 'metric'}, 'id': 'call_

In [22]:
# 한국 기준 현재 날짜/시간을 반환하는 Tool
from datetime import datetime
from pytz import timezone

@tool
def get_current_datetime(format: str='%Y-%m-%d %H:%M:%S') -> str:
    """
    한국기준 현재시각정보를 문자열로 반환하는 Tool
    Args:
        format: 날짜/시각 형식 지정
    Return:
        현재시각 문자열

    get_current_datetime() -> "2026-01-15 12:18:32"
    """
    kst = timezone('Asia/Seoul')  # 한국 시간대(KST) 설정
    return datetime.now(kst).strftime(format)  # 현재 서울 시간을 받아, format 형식의 문자열로 반환

get_current_datetime

StructuredTool(name='get_current_datetime', description='한국기준 현재시각정보를 문자열로 반환하는 Tool\nArgs:\n    format: 날짜/시각 형식 지정\nReturn:\n    현재시각 문자열\n\nget_current_datetime() -> "2026-01-15 12:18:32"', args_schema=<class 'langchain_core.utils.pydantic.get_current_datetime'>, func=<function get_current_datetime at 0x00000280594031A0>)

In [23]:
@tool
def calculate_age(today_date: str, birth_date: str) -> int:
    """
    오늘날짜, 생년월일을 입력받아 만나이를 계산하는 Tool
    Args:
        - today_date(str): 오늘 날짜 (yyyy-mm-dd형식)
        - birth_date(str): 생년월일 (yyyy-mm-dd형식)
    Return:
        - 계산된 만나이(int)
    """
    try:
        today = datetime.strptime(today_date, '%Y-%m-%d')  # 오늘 날짜 문자열 -> datetime 변환
        birthday = datetime.strptime(birth_date, '%Y-%m-%d')  # 생일 날짜 문자열 -> datetime 변환

        age = today.year - birthday.year  # 기본 나이 계산
        # 생일이 아직 안지난 경우
        if (today.month, today.day) < (birthday.month, birthday.day):
            age -= 1  # 만나이는 -1
        return age
    except ValueError:
        return "날짜 형식이 올바르지 않습니다. yyyy-mm-dd 형식으로 전달해 주세요."

calculate_age.invoke({'today_date': '2026-08-27', 'birth_date': '2020-10-11'})

5

In [24]:
llm = init_chat_model('gpt-5.4-mini')

tools = load_tools(['wikipedia']) + [get_current_datetime, calculate_age]

agent = create_agent(llm, tools, system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.')

response = agent.invoke(
    {'messages': [('human', '트럼프 대통령의 현재 나이는?')]},
    config = {'recursion_limit': 20}  # ReAct 멀티턴 (툴 호출 반복) 최대횟수 제한
)

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='트럼프 대통령의 현재 나이는?', additional_kwargs={}, response_metadata={}, id='0fe9b6dd-0fd9-4b50-8304-708cdfe5906c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 378, 'total_tokens': 395, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeUwOiqV60jANwpp2AOWZ6o8zQMv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b6-1c9b-7e82-9ec2-044497c7d001-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Donald Trump'}, 'id': 'call_CdF6Kvr99AIsJZa8TaYVmaUm', 

## Memory
agent의 checkpointer속성에 메모리객체를 대화내역을 저장한다.
- 임시저장 InMemorySaver()
- 영구저장 SqliteSaver()

### InMemorySaver

In [25]:
from langgraph.checkpoint.memory import InMemorySaver  # 메모리 기반 체크포인터 (대화 상태 저장)

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

# 체크포인터를 전달하여 대화 상태 저장 가능하도록 에이전트 생성
agent = create_agent(llm, tools, checkpointer=InMemorySaver())

response = agent.invoke(
    input = {'messages': [('human', '안녕! 만나서 반갑다! 나는 cap이라고 해. 넌 누구니?')]},
    config = {'configurable': {'thread_id': '100'}}  # thread_id로 대화 식별
)

print(response['messages'][-1].content)

안녕 cap! 만나서 반가워 😊  
나는 ChatGPT라고 해. 질문에 답하거나, 글을 쓰거나, 아이디어를 정리하는 걸 도와주는 AI야.  

원하면 편하게 말 걸어줘!


In [26]:
response = agent.invoke(
    input = {'messages': [('human', '어 그래 너 GPT구나~ 내 이름이 뭐였지? 나 기억상실증이야!')]},
    config = {'configurable': {'thread_id': '100'}}  # thread_id 100번으로 대화 유지
)

print(response['messages'][-1].content)

네 이름은 **cap**이야.


In [27]:
response = agent.invoke(
    input = {'messages': [('human', '어 그래 너 GPT구나~ 내 이름이 뭐였지? 나 기억상실증이야!')]},
    config = {'configurable': {'thread_id': '200'}}  # thread_id 200번으로 새로운 대화 시작
)

print(response['messages'][-1].content)

미안, 나는 지금 대화만 보고는 네 이름을 알 수 없어.  
이전에 네가 이름을 말했더라도 여기엔 기억이 남아있지 않아서 그래.

원하면 내가 기억하기 쉽게:
- “내 이름은 ○○야”라고 다시 알려줘
- 그러면 이 대화에서는 그 이름으로 불러줄게

혹시 원하면 내가 이름을 추측해볼 수도 있는데, 정확하진 않아 🙂


In [28]:
response = agent.invoke(
    input = {'messages': [('human', '내 이름은 아무것도 아니다. 꼭 기억해줘.')]},
    config = {'configurable': {'thread_id': '200'}}  # thread_id 200번으로 새로운 대화 시작
)

print(response['messages'][-1].content)

알겠어. 이 대화에서는 **“아무것도 아니다”**라고 기억하고 부를게.


In [29]:
response = agent.invoke(
    input = {'messages': [('human', '내 이름이 뭐라고?')]},
    config = {'configurable': {'thread_id': '200'}}  # thread_id 200번으로 새로운 대화 시작
)

print(response['messages'][-1].content)

네 이름은 **아무것도 아니다**야.


### sqliteSaver

In [30]:
# Langgraph 상태 저장을 SQLite로 영속화하여 저장하는 체크포인터 패키지
%pip install -Uqqq langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.


In [31]:
from langgraph.checkpoint.sqlite import SqliteSaver  # Sqlite 기반 체크포인트(Saver)
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

# Sqlite DB 연결을 checkpoint.db 컨텍스트로 관리
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()  # 테이블 생성 및 초기화

    # 에이전트가 상태 저장소로 checkpointer 활용
    agent = create_agent(llm, tools, checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human', 'Langchain에 대해 설명해줘.')]},
        config = {'configurable': {'thread_id': '100'}}  # thread_id로 대화 식별
    )

    pprint(response)
    print("=" * 50)
    pprint(response['messages'][-1].content)
    print("=" * 100)

    response = agent.invoke(
        input = {'messages': [('human', 'Langgraph에 대해 설명해줘.')]},
        config = {'configurable': {'thread_id': '100'}}  # thread_id로 대화 식별
    )

    pprint(response)
    print("=" * 50)
    pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='Langchain에 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='65bde86f-3b5f-4d12-9fcd-404c089c9eb8'),
              AIMessage(content='LangChain은 **대규모 언어 모델(LLM)을 활용한 애플리케이션을 쉽게 만들기 위한 프레임워크**예요.  \nChatGPT 같은 모델을 단순히 “질문-답변” 용도로 쓰는 것을 넘어서, **검색, 메모리, 도구 사용, 여러 단계의 작업 흐름**과 연결해 복잡한 앱을 만들 수 있게 도와줍니다.\n\n## 한 줄 요약\n**LangChain = LLM을 중심으로 한 AI 애플리케이션 개발용 도구 모음**\n\n---\n\n## 왜 필요한가?\nLLM만 단독으로 쓰면 이런 한계가 있어요.\n\n- 최신 정보에 약함\n- 외부 데이터베이스나 문서와 연결이 어려움\n- 여러 단계 작업을 자동화하기 까다로움\n- 대화 맥락을 오래 유지하기 어려움\n- API 호출, 검색, 계산 같은 외부 도구와의 연동이 번거로움\n\nLangChain은 이런 문제를 해결하려고 만들어졌습니다.\n\n---\n\n## 주요 기능\n### 1. 프롬프트 관리\nLLM에 보낼 입력을 구조화하고 재사용하기 쉽게 해줍니다.\n\n### 2. 체인(Chain)\n여러 단계를 순서대로 연결할 수 있어요.  \n예:\n- 문서 요약\n- 요약 결과를 기반으로 질의응답\n- 결과를 다시 정리\n\n### 3. 검색/문서 연결\nPDF, 웹페이지, DB 같은 외부 지식과 연결해서  \n**RAG(Retrieval-Augmented Generation)** 형태로 활용할 수 있습니다.\n\n### 4. 메모리\n대화 내용을 저장해 이전 맥락을 반영할 수 있게 합니다.\n\n### 5. 에이전트(Agent)\nLLM이 상황에 따라\n- 검색 도구\n- 계산기\n- DB\n- API\n\n같

In [32]:
# 사용자가 재접속한 상황
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()  # 테이블 생성 및 초기화 (기존에 존재하면 그대로 사용)

    # 에이전트가 상태 저장소로 checkpointer 활용
    agent = create_agent(llm, tools, checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human', '오케이. 완전 이해했어! 그럼 니가 말해준 langchain, langgraph를 세줄요약해줘.')]},
        config = {'configurable': {'thread_id': '100'}}  # thread_id로 대화 식별
    )

    pprint(response)
    print("=" * 50)
    pprint(response['messages'][-1].content)
    print("=" * 100)

{'messages': [HumanMessage(content='Langchain에 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='65bde86f-3b5f-4d12-9fcd-404c089c9eb8'),
              AIMessage(content='LangChain은 **대규모 언어 모델(LLM)을 활용한 애플리케이션을 쉽게 만들기 위한 프레임워크**예요.  \nChatGPT 같은 모델을 단순히 “질문-답변” 용도로 쓰는 것을 넘어서, **검색, 메모리, 도구 사용, 여러 단계의 작업 흐름**과 연결해 복잡한 앱을 만들 수 있게 도와줍니다.\n\n## 한 줄 요약\n**LangChain = LLM을 중심으로 한 AI 애플리케이션 개발용 도구 모음**\n\n---\n\n## 왜 필요한가?\nLLM만 단독으로 쓰면 이런 한계가 있어요.\n\n- 최신 정보에 약함\n- 외부 데이터베이스나 문서와 연결이 어려움\n- 여러 단계 작업을 자동화하기 까다로움\n- 대화 맥락을 오래 유지하기 어려움\n- API 호출, 검색, 계산 같은 외부 도구와의 연동이 번거로움\n\nLangChain은 이런 문제를 해결하려고 만들어졌습니다.\n\n---\n\n## 주요 기능\n### 1. 프롬프트 관리\nLLM에 보낼 입력을 구조화하고 재사용하기 쉽게 해줍니다.\n\n### 2. 체인(Chain)\n여러 단계를 순서대로 연결할 수 있어요.  \n예:\n- 문서 요약\n- 요약 결과를 기반으로 질의응답\n- 결과를 다시 정리\n\n### 3. 검색/문서 연결\nPDF, 웹페이지, DB 같은 외부 지식과 연결해서  \n**RAG(Retrieval-Augmented Generation)** 형태로 활용할 수 있습니다.\n\n### 4. 메모리\n대화 내용을 저장해 이전 맥락을 반영할 수 있게 합니다.\n\n### 5. 에이전트(Agent)\nLLM이 상황에 따라\n- 검색 도구\n- 계산기\n- DB\n- API\n\n같

In [ ]:
# SQLite 체크포인터(DB)의 특정 thread_id의 대화 메시지 조회
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    # thread_id가 100인 체크포인트 조회
    checkpointer_tuple = checkpointer.get_tuple({"configurable": {"thread_id": "100"}})

    checkpointer_data = checkpointer_tuple.checkpoint  # 원본 데이터 dict
    messages = checkpointer_data['channel_values']['messages']  # 저장된 메시지 목록

    for i, message in enumerate(messages, 1):
        # message.type속성값이 있으면 사용, 없으면 클래스명 사용
        msg_type = getattr(message, 'type', message.__class__.__name__)
        print(f"{i}: [{msg_type}] {message.content}")  # 번호/타입/내용
        print()

1: [human] Langchain에 대해 설명해줘.

2: [ai] LangChain은 **대규모 언어 모델(LLM)을 활용한 애플리케이션을 쉽게 만들기 위한 프레임워크**예요.  
ChatGPT 같은 모델을 단순히 “질문-답변” 용도로 쓰는 것을 넘어서, **검색, 메모리, 도구 사용, 여러 단계의 작업 흐름**과 연결해 복잡한 앱을 만들 수 있게 도와줍니다.

## 한 줄 요약
**LangChain = LLM을 중심으로 한 AI 애플리케이션 개발용 도구 모음**

---

## 왜 필요한가?
LLM만 단독으로 쓰면 이런 한계가 있어요.

- 최신 정보에 약함
- 외부 데이터베이스나 문서와 연결이 어려움
- 여러 단계 작업을 자동화하기 까다로움
- 대화 맥락을 오래 유지하기 어려움
- API 호출, 검색, 계산 같은 외부 도구와의 연동이 번거로움

LangChain은 이런 문제를 해결하려고 만들어졌습니다.

---

## 주요 기능
### 1. 프롬프트 관리
LLM에 보낼 입력을 구조화하고 재사용하기 쉽게 해줍니다.

### 2. 체인(Chain)
여러 단계를 순서대로 연결할 수 있어요.  
예:
- 문서 요약
- 요약 결과를 기반으로 질의응답
- 결과를 다시 정리

### 3. 검색/문서 연결
PDF, 웹페이지, DB 같은 외부 지식과 연결해서  
**RAG(Retrieval-Augmented Generation)** 형태로 활용할 수 있습니다.

### 4. 메모리
대화 내용을 저장해 이전 맥락을 반영할 수 있게 합니다.

### 5. 에이전트(Agent)
LLM이 상황에 따라
- 검색 도구
- 계산기
- DB
- API

같은 도구를 스스로 선택해 쓰게 할 수 있습니다.

---

## 예시
예를 들어 고객 상담 챗봇을 만든다고 하면:

1. 사용자가 질문함
2. LangChain이 질문을 분석
3. 필요한 경우 사내 문서 검색
4. 관련 내용을 LLM에 전달
5. 답변 생성
6. 대화 기록 저장

이런 흐름을 쉽게 구성할 수 있습니다.

---



1️⃣ 세션(메모리) 유지 방식
예: store = {}, ChatMessageHistory, InMemorySaver
- 특징
    - 서버 메모리에만 대화 상태를 저장
    - 서버 재시작/재배포 시 모두 사라짐
    - 구현이 가장 단순하고 빠름
- 사용 시기
    - 실습 / 데모 / PoC
    - 단일 서버, 짧은 대화
    - “지금 이 세션에서만 기억하면 되는” 경우
- 장단점
    - ✅ 속도 빠름, 구현 쉬움
    - ❌ 서버 내려가면 기억 소멸
    - ❌ 멀티 서버(스케일아웃) 불가능

2️⃣ SQLite 체크포인터
예: SqliteSaver, checkpoint.db
- 특징
    - 로컬 파일(DB)에 대화 상태 저장
    - 서버 재시작해도 대화 복원 가능
    - 설정/운영 부담이 거의 없음
- 사용 시기
    - 1대 서버 운영
    - “재접속 시 대화 이어가기”가 중요한 서비스
    - 내부 도구, 사내용 챗봇, 파일 기반 서비스
- 장단점
    - ✅ 재시작해도 대화 유지
    - ✅ 설정 간단 (파일 하나)
    - ❌ 동시접속/대량 트래픽에 취약
    - ❌ 운영·분석·확장성 한계

3️⃣ RDB (MySQL / PostgreSQL 등)
실무에서 가장 많이 쓰는 방식

- 특징
    - 대화 내역을 정규화된 테이블로 저장
    - 여러 서버가 공유 DB 사용 가능
    - 사용자/세션/대화/이력 분석까지 가능
- 사용 시기
    - 실서비스(운영 환경)
    - 로그인 사용자 기반 챗봇
    - 고객지원, 상담, 금융, 헬스케어, 교육 서비스
- 장단점
    - ✅ 서버 여러 대에서도 동일한 대화 유지
    - ✅ 로그/분석/감사/리포트 가능
    - ✅ 권한·보안·백업 체계화 가능
    - ❌ 설계/운영 비용 존재

- 요약하자면  
메모리 세션	: 빠르고 간단  
SQLite 같은 파일형 DB : 재접속 기억 + 운영 부담 최소  
RDB	같은 관계형 데이터베이스 : 확장성, 안정성, 분석, 운영  

- 서비스 구상 단계에서  
사용자별 히스토리 관리  
문제 발생 시 감사 로그  
대화 품질/모델 성능 분석  
요약/임베딩/재검색(RAG) 연계  
개인화 서비스(추천, 성향 파악)  
이걸 하려면 RDB 또는 그 이상(이벤트 로그, 데이터 웨어하우스) 가 필요

## Middleware
https://docs.langchain.com/oss/python/langchain/middleware

미들웨어를 통해 에이전트의 추론 과정 중간에 개입하여 내부 동작을 커스터마이징할 수 있다.
Agent를 세부적으로 커스터마이징하기 위한 대부분의 작업을 미들웨어로 할 수 있다.

- 대화 기록 요약
- 동작 중 사용자 입력 대기
- 특정 모델 또는 tool에 대한 호출 제약
- fallback
- PII(개인식별정보) 처리 등

### SummarizationMiddleware

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware  # 대화내역 자동 요약 미들웨어

model = init_chat_model('gpt-5.4-mini')  # 메인 에이전트 LLM (상대적으로 성능이 좋고 비싼모델)
summary_model = init_chat_model('gpt-5.4-mini')  # 요약 모델 (상대적으로 싼모델)

middelware_summarize = SummarizationMiddleware(
    model = summary_model,       # 요약모델
    trigger = ('tokens', 1000),  # 누적 노큰 1000 이상일때 트리거
    keep = ('messages', 1),      # 최근 1개 메시지는 요약 제외
    summary_prompt = '다음 대화 내용을 적절하게 요약해주세요.\n{messages}'
)

agent = create_agent(
    model = model,
    tools = [],
    checkpointer = InMemorySaver(),  # 메모리 저장소
    middleware = [middelware_summarize]  # 요약 미들웨어 사용
)

In [35]:
response = agent.invoke(
    input = {'messages':[('human', '뮤지컬 Wicked의 내용을 Elphaba 입장에서 서술해줘. Elphaba역으로 연극에 출연해야되서 준비중이야. 중요한 포인트들을 다 짚어줘!')]},
    config = {'configurable': {'thread_id': '5'}}
)

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)
print("=" * 100)

response = agent.invoke(
    input = {'messages':[('human', 'Elphaba가 Glinda를 어떤 심정으로 보는게 좋을까?')]},
    config = {'configurable': {'thread_id': '5'}}
)

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)
print("=" * 100)

response = agent.invoke(
    input = {'messages':[('human', '또 메소드 연기가 필요한 부분이 있을까?')]},
    config = {'configurable': {'thread_id': '5'}}
)

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)
print("=" * 100)

{'messages': [HumanMessage(content='뮤지컬 Wicked의 내용을 Elphaba 입장에서 서술해줘. Elphaba역으로 연극에 출연해야되서 준비중이야. 중요한 포인트들을 다 짚어줘!', additional_kwargs={}, response_metadata={}, id='828f9d76-79d6-4b17-bd2e-9ebe0819ccf8'),
              AIMessage(content='물론이야.  \n**뮤지컬 *Wicked*를 Elphaba(엘파바) 입장에서 이해하기 쉽게, 연기 준비용으로 중요한 포인트 중심으로 정리**해줄게.  \nElphaba는 단순히 “초록 피부의 마녀”가 아니라, **세상에서 오해받고 밀려났지만 끝까지 자기 신념을 지키는 인물**이야. 이 관점을 잡으면 연기가 훨씬 살아나.\n\n---\n\n# 1) Elphaba의 핵심 인물 해석\n\nElphaba는 보통 이렇게 보이지만:\n\n- 겉으로는 차갑고 예민해 보임\n- 하지만 실제로는 **굉장히 상처가 많고**\n- 정의감이 강하며\n- 사랑받고 싶지만 잘 표현하지 못하고\n- 자기와 다른 사람들을 지키려는 마음이 큼\n\n즉, Elphaba는 “악당”이 아니라  \n**사랑과 정의를 원했지만, 세상이 그녀를 악역으로 만들었던 사람**이야.\n\n연기할 때 가장 중요한 건:\n- “나는 왜 이렇게 예민하지?”가 아니라\n- **“나는 왜 아무도 진심을 보지 못해주지?”**\n라는 감정이 바탕에 있어야 해.\n\n---\n\n# 2) Elphaba의 이야기 흐름: 장면별로 보기\n\n## A. 어린 시절과 출생\nElphaba는 태어날 때부터 **초록 피부**를 가지고 태어나.  \n이건 단순한 외모 특징이 아니라, 그녀의 인생 전체를 규정하는 낙인이야.\n\n### 감정 포인트\n- 태어나자마자 “다르다”는 이유로 거리감과 시선을 받음\n- 부모의 사랑도 안정적이지 않음\n- 특히 아버지는 Elphaba를 불편하게 여김\n- 그래서 Elphab

### PIIMiddleware
**PII (Personally Identifiable Information) 개인식별정보 처리**

https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection

In [ ]:
from langchain.agents.middleware import PIIMiddleware  # PII(개인정보) 탐지/처리 미들웨어

middelware_email = PIIMiddleware(
    pii_type = 'email',  # 이메일 PII 처리
    detector = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", # +: 1글자이상, {2,}: 2글자이상
    strategy = 'redact',  # 삭제처리
    apply_to_input = True  # 사용자 입력에 대해 적용
)

middelware_credit_card = PIIMiddleware(
    pii_type = 'credit_card',  # 신용카드 PII 처리
    detector = r'(?:\d{4}[-\s]?){3}\d{4}|\d{4}[-\s]?\d{6}[-\s]?\d{5}',  # 카드번호 16자리 표현식
    strategy = 'mask',  # 마스킹 처리
    apply_to_input = True
)

middelware_api_key = PIIMiddleware(
    pii_type = 'api_key',  # API 키 PII 처리
    detector = r"sk-[a-zA-Z0-9-_]{161}",  # OpenAI 키 예시
    strategy = 'mask'
)

agent = create_agent(
    model = init_chat_model('gpt-5.4-mini'),
    tools = [],
    middleware = [middelware_email, middelware_credit_card, middelware_api_key]
)

In [39]:
response = agent.invoke({
    'messages': [('human', 'AWS를 사용하는데 요금이 잘못 청구된 것 같아. 그래서 금액 재조정을 요청하는 메일을 작성하려고 해. 영어로 메일내용을 작성해줘. 내 이메일은 capybara@gmail.com이야.')]
})

pprint(response)

{'messages': [HumanMessage(content='AWS를 사용하는데 요금이 잘못 청구된 것 같아. 그래서 금액 재조정을 요청하는 메일을 작성하려고 해. 영어로 메일내용을 작성해줘. 내 이메일은 [REDACTED_EMAIL]이야.', additional_kwargs={}, response_metadata={}, id='196ab810-c577-4337-ba9b-b2109140f096'),
              AIMessage(content='물론입니다. 아래는 AWS 청구 오류로 보이는 금액에 대해 **재조정 요청**을 하는 영어 이메일 예시입니다.  \n이메일 주소는 요청하신 대로 넣었습니다.\n\n---\n\n**Subject:** Request for Billing Review and Adjustment\n\nDear AWS Billing Support Team,\n\nI hope you are doing well.\n\nI am writing to request a review of my AWS bill, as I believe there may have been an incorrect charge on my account. I would appreciate it if you could investigate the issue and adjust the amount if any billing error is confirmed.\n\nMy account email address is **[REDACTED_EMAIL]**.\n\nPlease let me know if you need any additional information, such as invoice details, dates, or related resource IDs. I would be happy to provide anything needed to help resolve this matter quickly.\n\nThank you for your time and suppo

In [40]:
response = agent.invoke({
    'messages': [('human', '환불요청 메일 작성해줘. 내 신용카드 번호는 1234-1234-1234-0000이야. 최근에 쿠팡에서 주문한 100만원짜리 피규어 환불처리 요청해줘.')]
})

pprint(response)

{'messages': [HumanMessage(content='환불요청 메일 작성해줘. 내 신용카드 번호는 ****-****-****-0000이야. 최근에 쿠팡에서 주문한 100만원짜리 피규어 환불처리 요청해줘.', additional_kwargs={}, response_metadata={}, id='f29f1603-5c73-4b56-baf9-bf63a433c9d6'),
              AIMessage(content='물론이에요. 다만 **신용카드 번호는 메일에 직접 적지 않는 게 안전**합니다. 보통은 **주문번호, 주문자명, 연락처, 결제수단(카드사명만)** 정도로 충분해요.\n\n아래처럼 보내시면 됩니다:\n\n---\n\n**제목:** 쿠팡 주문 상품 환불 요청드립니다\n\n안녕하세요.  \n최근 쿠팡에서 주문한 **100만원 상당의 피규어**에 대해 환불을 요청드립니다.\n\n- 주문내역: 쿠팡 주문 피규어\n- 주문금액: 1,000,000원\n- 환불사유: [여기에 사유 입력]\n- 주문자명: [이름 입력]\n- 주문번호: [주문번호 입력]\n\n환불 절차 안내 부탁드리며, 필요하신 추가 정보가 있으면 알려주시기 바랍니다.  \n확인 후 빠른 처리 부탁드립니다.\n\n감사합니다.  \n[이름]  \n[연락처]\n\n---\n\n원하시면 제가 바로  \n1) **더 공손한 버전**,  \n2) **강하게 요청하는 버전**,  \n3) **쿠팡 고객센터에 보내는 짧은 버전**  \n중 하나로 다시 써드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 276, 'prompt_tokens': 58, 'total_tokens': 334, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0

In [41]:
response = agent.invoke({
    'messages': [('human', '환경변수 설정을 잘못한 것 같아. .env파일 내용좀 봐줘. OPENAI_API_KEY = sk-proj-Ax5O_j60oh6DjSBkdbo2VSeRswbbSocYNtaStOBGNfo4e0toXk1cQpxb9Lub32wmyTxK8FhDWdT3BdbkFJIbdnY9sVHe0Oh6WBpzcRqDBRqxUqLwTm1MX21q0Ih6LL96dIHfdrFeN2xS6RZ2WH1hJ3RlnfIA')]
})

pprint(response)

{'messages': [HumanMessage(content='환경변수 설정을 잘못한 것 같아. .env파일 내용좀 봐줘. OPENAI_API_KEY = ****nfIA', additional_kwargs={}, response_metadata={}, id='6e6777be-f730-424a-b289-5c060fbfacbc'),
              AIMessage(content='`.env`에 적는 방식은 보통 아래처럼 **공백 없이** 써야 합니다.\n\n```env\nOPENAI_API_KEY=****nfIA\n```\n\n지금처럼\n\n```env\nOPENAI_API_KEY = ****nfIA\n```\n\n라고 쓰면, 일부 로더에서는 **변수 이름에 공백이 포함된 것처럼** 인식해서 문제가 날 수 있어요.\n\n추가로 확인할 것:\n- 값 앞뒤에 **따옴표가 필요한지**: 보통 키는 안 넣어도 됨\n- `.env` 파일 맨 위/아래에 **불필요한 공백이나 특수문자**가 없는지\n- 코드에서 `.env`를 제대로 읽는지 (`dotenv` 같은 라이브러리 사용 여부)\n\n예:\n```env\nOPENAI_API_KEY=sk-...\n```\n\n원하시면 `.env` 전체 형식도 안전하게 점검해드릴게요.  \n다만 **실제 키 전체는 절대 보내지 마세요**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 223, 'prompt_tokens': 35, 'total_tokens': 258, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio

## Streaming
*openai모델은 조직인증된 사용자에 한해서 stream기능을 사용할수 있다.*

In [43]:
model = init_chat_model('gpt-5.4-mini')
agent = create_agent(model)
# 스트리밍 실행 (메시지 단위로 받음)
stream = agent.stream(
    input = {'messages': [('human', '역사상 가장 위대한 한국인은 누구인가요? 10명과 점수까지 주세요.')]},
    stream_mode= 'messages'
)

for chunk, metadata in stream:  # chunk(메시지 조각)와 metadata(정보)를 순회
    print(chunk.content, end='', flush=True)  # 조각을 이어붙이면서 출력

“역사상 가장 위대한 한국인”은 **기준에 따라 달라집니다.**  
예를 들어 **국가에 끼친 영향, 세계적 업적, 문화적 상징성, 윤리적 평가, 역사적 파급력**을 종합해 보면 순위는 달라질 수 있어요.

아래는 **주관적이지만 비교적 균형 있게** 뽑아본 **한국 역사상 위대한 인물 10명**과 **100점 만점 기준 점수**입니다.

## 한국 역사상 위대한 인물 TOP 10

| 순위 | 인물 | 점수 | 간단한 이유 |
|---|---|---:|---|
| 1 | **세종대왕** | **100** | 한글 창제, 문화·과학·정치 전반의 업적이 압도적 |
| 2 | **이순신** | **98** | 임진왜란에서 국가를 구한 전략가이자 상징적 영웅 |
| 3 | **김구** | **95** | 독립운동의 상징, 민족 정체성과 통합의 상징성 큼 |
| 4 | **퇴계 이황** | **92** | 조선 성리학의 대표, 한국 지성사에 큰 영향 |
| 5 | **율곡 이이** | **91** | 개혁·정치·학문 모두에서 뛰어난 사상가 |
| 6 | **강감찬** | **89** | 귀주대첩으로 고려를 지킨 명장 |
| 7 | **장보고** | **88** | 해상무역과 동아시아 네트워크를 주도한 인물 |
| 8 | **신사임당** | **86** | 예술·교육의 상징, 한국 문화 이미지에 큰 영향 |
| 9 | **안중근** | **85** | 독립운동의 상징적 인물, 동아시아 역사에도 영향 |
| 10 | **손기정** | **82** | 식민지 시대를 넘어 세계무대에서 한국인의 존재를 알림 |

## 제 의견으로 1위는?
제가 하나만 꼽는다면 **세종대왕**입니다.  
이유는 단순히 훌륭한 왕이라서가 아니라, **한글 창제라는 독보적 업적이 오늘날 한국인의 언어·문해력·문화 정체성 전체를 바꿨기 때문**입니다. 영향력이 **현재까지도 살아 있는** 점이 매우 큽니다.

## 참고로
- **영웅성** 기준이면: **이순신, 김구, 안중근**
- **문화·문명 